<a href="https://colab.research.google.com/github/0-ahmesmail-0/my-flyrank-repo/blob/main/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
'''
I am pursuing Lane 2: Refresh / Content Opportunity Scoring. From a Machine Learning systems perspective, this problem
 is framed as a Scoring and Ranking task. Instead of relying on rigid, binary classifications, the system will output a continuous
 score for each URL, allowing us to generate a prioritized, ranked review queue for the content and SEO teams.

'''

'\nI am pursuing Lane 2: Refresh / Content Opportunity Scoring. From a Machine Learning systems perspective, this problem\n is framed as a Scoring and Ranking task. Instead of relying on rigid, binary classifications, the system will output a continuous\n score for each URL, allowing us to generate a prioritized, ranked review queue for the content and SEO teams.\n \n'

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
'''
The ideal target is the future value at risk (e.g., predicted search traffic decline or
potential recovery volume over the next 30 days based on prior 90-day features). In this starter phase,
we utilize a binary proxy label representing traffic decay: is_declining_label = (trend_direction == 'down').
This proxy acts as a reliable stand-in for content weakness, indicating that a page is actively losing organic
traction and requires human intervention.
'''

"\nThe ideal target is the future value at risk (e.g., predicted search traffic decline or \npotential recovery volume over the next 30 days based on prior 90-day features). In this starter phase, \nwe utilize a binary proxy label representing traffic decay: is_declining_label = (trend_direction == 'down').\nThis proxy acts as a reliable stand-in for content weakness, indicating that a page is actively losing organic\ntraction and requires human intervention.\n"

## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
'''
Since the output of this model is a prioritized queue meant to support a human editorial team
with limited time capacity, traditional global accuracy is not the right metric. Instead, our
primary success metrics are:
Precision@50 (or Precision@K): Out of the top 50 high-priority pages recommended for a
refresh, how many were actually declining or at risk?
Average Precision (AP): To evaluate the quality and sorting correctness of the entire ranked list.
'''

'\nSince the output of this model is a prioritized queue meant to support a human editorial team \nwith limited time capacity, traditional global accuracy is not the right metric. Instead, our \nprimary success metrics are:  \nPrecision@50 (or Precision@K): Out of the top 50 high-priority pages recommended for a \nrefresh, how many were actually declining or at risk?  \nAverage Precision (AP): To evaluate the quality and sorting correctness of the entire ranked list.  \n'

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
'''
The unit of analysis (grain) is one unique pseudonymized content item (one page/URL) per client.
In the code cell below, I load the starter dataset to demonstrate this grain. Each row represents the historical search,
traffic, and structural features of a single web page over a fixed window.
'''
import pandas as pd
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # move from notebooks/ to the repo root

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")
# Load the starter data to display the unit of analysis
df = pd.read_csv("../data/raw/content_refresh_anonymized.csv")

# Display the grain: one row per content item
print(f"DataFrame Grain Check: {df.shape[0]} rows, where each row = 1 unique page.")
df[['content_id', 'client_id', 'impressions_90d', 'sessions_90d', 'trend_direction']].head()

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. You're ready.
DataFrame Grain Check: 30000 rows, where each row = 1 unique page.


,content_id,client_id,impressions_90d,sessions_90d,trend_direction
0,content_304f48230142,client_f369cb89fc,3803,17,down
1,content_a1fb4e703a9e,client_4e07408562,15320,9,down
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,down
3,content_331d6c4de07b,client_19581e27de,11751,78,stable
4,content_d99b7a2d90ca,client_3fdba35f04,19140,145,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
'''
Fixed, rule-based logic (e.g., "refresh any page where clicks drop by 20%") is too rigid
and suffers from significant noise. A static rule cannot naturally balance search volume
\against relative decline, nor can it filter out look-alike patterns such as natural
seasonality, search engine result page (SERP) layout changes, or traffic consolidation across
internal sibling pages. Machine Learning beats a fixed rule because it can scale to ingest
multiple historical signals simultaneously (impressions, position tiers, engagement, and age)
to uncover non-linear patterns, effectively separating high-value decay risks from low-volume background noise.
'''

'\nFixed, rule-based logic (e.g., "refresh any page where clicks drop by 20%") is too rigid \nand suffers from significant noise. A static rule cannot naturally balance search volume \n\x07gainst relative decline, nor can it filter out look-alike patterns such as natural \nseasonality, search engine result page (SERP) layout changes, or traffic consolidation across \ninternal sibling pages. Machine Learning beats a fixed rule because it can scale to ingest \nmultiple historical signals simultaneously (impressions, position tiers, engagement, and age) \nto uncover non-linear patterns, effectively separating high-value decay risks from low-volume background noise.\n'

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.